# 📚 20.5 ResNet — COMPLETE NOTES

> **Core idea:**
> **Degradation problem → Residual learning → Skip connection → Better information/gradient flow → Easier optimization of very deep networks**

The original ResNet paper introduced residual learning specifically to make substantially deeper networks easier to optimize and demonstrated networks up to 152 layers on ImageNet. ([arXiv][1])

---

# 1. 🎯 What is ResNet?

**ResNet = Residual Network.**

The central idea is:

> Instead of forcing a group of layers to learn the complete desired transformation, let them learn the **residual correction** and add the original input through a shortcut.

Mathematically:

$$
\boxed{y=F(x)+x}
$$

where:

* \(x\) = input
* \(F(x)\) = learned residual function
* \(y\) = output

The original paper calls this **residual learning**. ([arXiv][1])

---

# 2. 🚨 The Degradation Problem

## Why not simply keep adding layers?

Suppose:

```text
20-layer CNN
       ↓
Good training performance
```

We might expect:

```text
56-layer CNN
       ↓
More layers
       ↓
More learning capacity
       ↓
Even better training performance
```

But the original ResNet work showed something surprising:

> A deeper **plain** network could have **higher training error** than a shallower one. ([arXiv][1])

This is called the:

# 🔥 Degradation Problem

### Important:

This is **not simply overfitting**.

Overfitting typically looks like:

```text
Training error ↓
Test error ↑
```

The degradation problem can look like:

```text
Training error ↑
```

when the plain network becomes deeper.

So the problem is fundamentally about **optimization difficulty**, not merely lack of generalization. ([arXiv][1])

---

# 🖼️ 3. Degradation Problem — Visual

![Image](https://images.openai.com/static-rsc-4/rxErlLi30FssFYAQxC2xc3bcJWt5w189p2LvHOVHTvG-KD2Xu4VVtRXHnRQHcI2sjrzl5U9K7BTIWVwPfD4bNQtRMrOZrgG8cPNjUX_Vo0F9Lic3fi-mv7fWtPGnRLczJTR5cJsCUMiSYf4cULLb-IntdHD8_wq2nd_H-jwLCvpvgUfbhkS5L3u8qd8StftY?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/QEvGYI1IwJShY4wVDJByvdCGhLvG-UeSDYFNH3OIK_2ubWh8v6KJB5Bn3kuXIqAvS74XBZRGjhamZ5ZmcnbC1Fl5D0NKAE2hEbAcRuYrbZncoaQHJ-QkE4hsEOC_GurN39GKmXT3nL-oSQvTgc9gpdG4otR1Ml7PgGfdG7bIz_dRVkotnLo3kAOMHPpNQYVw?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/ODCPjMdLV4PH2YI2-BPGz3YOVhrGjZCrdldkXzr3_fuwZ3aBq2sVARxIRBCNrvENLtHFwn6ThnNwXfNe3DdrxNAIHtMQj3gv_OSF-6vKXXlDmHxP8sHQzIzUcy-IXzyzoZr4czDJ0t4bt4dGhwW9RwOpZZLsdPZFB5PvThLCk3Zd9uA_NzYjt7a4ha2AEdsz?purpose=fullsize)

The mental picture:

```text
Training Error
     ↑
     │
     │       Plain network
     │          /
     │         /
     │________/
     │
     └──────────────────→ Network Depth
```

Initially:

```text
More depth
    ↓
Better
```

But eventually:

```text
More depth
    ↓
Harder optimization
    ↓
Higher training error
```

---

# 4. 🤔 Why Is This Surprising?

Imagine we already have a good network:

```text
Input
 ↓
20 useful layers
 ↓
Good representation
```

Now we add two layers.

In principle, those two layers could simply learn:

$$
H(x)=x
$$

meaning:

> **"Don't change the representation."**

If the deeper network could easily learn this identity mapping, adding layers shouldn't make the training result worse.

But plain deep networks can struggle to learn this easily.

This motivates **residual learning**.

---

# 5. 🧠 Residual Learning

Instead of directly learning:

$$
H(x)
$$

ResNet defines:

$$
F(x)=H(x)-x
$$

Therefore:

$$
\boxed{H(x)=F(x)+x}
$$

The network learns:

> **What modification should be made to the current representation?**

rather than necessarily learning:

> "What should the entire output representation be from scratch?"

---

# 6. 💡 Intuition

Think of:

```text
Current representation
        ↓
        x
```

The network asks:

```text
"What correction should I make?"
        ↓
       F(x)
```

Then:

```text
New representation

x + F(x)
```

So:

```text
                 Current information
                       │
                       ↓
                      x
                       │
                       ├────────────────┐
                       │                │
                       ↓                │
                 Learn correction       │
                       │                │
                      F(x)              │
                       │                │
                       └───────→ (+) ←──┘
                                  │
                                  ↓
                                y
```

---

# 7. 🔢 Numerical Example

Suppose:

$$
x=10
$$

and the residual branch learns:

$$
F(x)=3
$$

Then:

$$
y=F(x)+x
$$

$$
y=3+10
$$

$$
\boxed{y=13}
$$

The residual branch learned:

> **Increase the representation by 3.**

---

### Negative residual

Suppose:

$$
x=10
$$

and:

$$
F(x)=-2
$$

Then:

$$
y=10-2
$$

$$
\boxed{y=8}
$$

So residual learning can learn:

```text
Positive correction
        +
Negative correction
        +
Almost zero correction
```

---

# 8. 🔥 The Most Important Case: \(F(x)=0\)

Suppose the additional layers don't need to change the representation.

Then:

$$
F(x)=0
$$

Therefore:

$$
y=F(x)+x
$$

$$
y=0+x
$$

$$
\boxed{y=x}
$$

🔥 This means the residual block can represent an **identity mapping** naturally.

That's a crucial reason residual learning helps when building deeper networks.

---

# 9. 🧩 Skip Connection

The path carrying \(x\) directly around the convolutional layers is called a:

> **Skip connection** or **shortcut connection**

![Image](https://images.openai.com/static-rsc-4/dkFMnUrlSVBHrCJcHP-ZhzXsEcBD52_Q1WkiIXuNISwKKPVfeo-IwQF5MuwvvKsbS4nTqEfQHf589oQeOhJZcNBcFm98LJIfA1nxE7djlOQ8NGofBRXbFLuX8dDo0BmOOr_j0NAACgFCt9sIOYCc4p6UsAEOPdj1CgeAXy6aVQMBgQ1Dq6kLjqLdlcOQZyi4?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/xofa83cigxEIUbSQya82jV81PR8TDT-Btqfkan63FE0g7HdUOPy-MUZJoBV-ghALPA6f-_G5Wf1SugkQyzrSUIeJDe0GsL6mpLfbTlz3KXd0XiyP5zJzv2WlVvmYwMSPzeFWRQaXIjXie70ffNE9V9QNZx19ih08VKANGbqevwPbbrm-tRexHRqkoFcCjmgY?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/YdkPyghfveCpUwz7HhfNKn4c5TAcnPMPiI87I9fcbWFquWjaDyX6wqu0CnDww6eAX1uX6cMJ5RYlUuTDOBBf6XOlPvhsQboeoQ-jXfbazW1WPwrABSe8j3X7VunoRj95yeq_ZHqvbCyOjzzXs9y7WA4g_UEsokiOsP8WZWxJtv-4tbReAqMQrEf59XyP0LrK?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/OTcN1ExpxZw2Tx8cBBwgYDz3MzDtxHYaVUfjJSYmcPuEQijVriorOCjb7tYOGaidFy70t8KOR4zFJ6_5_K-yvPgCPh2mW1U_mCaRSaGt2ds2XcjVXMDFYExQeXUcmoAzAa1w1mAJSkuPk65-Vj0ZOy-Vlz8WhijbK4jJx-f2Pz2f1Aghf_VUgYPnmnSDVmEQ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/eB82Tf5HAue29O8UJrXrlY-p2Q3xJW18Icwce5MRZI6mDs1HZ1Geugz84d0pG_7huJHsj1zAZ6xCfYY_YUu5S2dhAE4Nx2VOnhY-jiLqc8HAowVJmSL8RH5nTdq2s7FdSGTAVxPb3YxW4_A1_1SJflxrNI6tvI2_LfCClOTDDqdrWQobp_EWocSSTO3i8ZnJ?purpose=fullsize)

Basic structure:

```text
                     ┌─────────────────┐
                     │                 │
                     │  Skip / Identity│
                     │                 ↓
Input x ─────────────┤              (+)
                     │                 ↑
                     ↓                 │
                  Conv                │
                     ↓                 │
                   ReLU                │
                     ↓                 │
                  Conv ────────────────┘
                                       ↓
                                     Output
```

Mathematically:

$$
\boxed{y=F(x)+x}
$$

The original paper describes the shortcut as an identity mapping in the simplest residual block. ([arXiv][1])

---

# 10. 🧠 Why "Skip"?

Because the input **skips over** some layers:

```text
Normal:

x
↓
Conv
↓
Conv
↓
Output
```

ResNet:

```text
                 ┌──────────────┐
                 │              │
x → Conv → Conv ─┴──────→ (+)
```

The shortcut bypasses the convolutional transformations.

---

# 11. 🔥 Residual Block

The fundamental ResNet unit is the:

# **Residual Block**

Simplified:

```text
                 ┌─────────────────────┐
                 │                     │
                 │       Identity      │
                 │                     ↓
x ───────────────┤                 ┌───────┐
                 │                 │   +   │
                 ↓                 └───┬───┘
              Conv 3×3                 │
                 ↓                     │
                ReLU                   │
                 ↓                     │
              Conv 3×3 ────────────────┘
                                       ↓
                                     Output
```

The residual branch learns:

$$
F(x)
$$

The shortcut carries:

$$
x
$$

Then:

$$
\boxed{y=F(x)+x}
$$

---

# 12. 🧠 Why Does This Help?

Compare the two approaches.

### Plain network

```text
x
 ↓
Layer
 ↓
Layer
 ↓
Layer
 ↓
y
```

The layers must learn the transformation directly.

### Residual network

```text
              ┌───────────────┐
              │               │
x ────────────┤               ↓
              │              (+)
              ↓               ↑
          F(x) layers ────────┘
```

The block only needs to learn:

$$
F(x)=H(x)-x
$$

So if the desired transformation is close to identity:

$$
F(x)\approx0
$$

the block can preserve the existing representation.

---

# 13. 🔥 Gradient Flow

This is probably the **most important mathematical part** of ResNet.

During training:

```text
Forward:
Input
 ↓
Layers
 ↓
Output
 ↓
Loss
```

Then backward:

```text
Loss
 ↓
Gradients
 ↓
Layers
 ↓
Earlier layers
```

Very deep plain networks can make gradient propagation difficult.

---

# 14. 📐 Residual Gradient Equation

Start with:

$$
y=F(x)+x
$$

Differentiate with respect to \(x\):

$$
\frac{\partial y}{\partial x}
=
\frac{\partial F(x)}{\partial x}
+
1
$$

Therefore:

$$
\boxed{
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\left(
\frac{\partial F}{\partial x}+1
\right)
}
$$

### Where did the \(+1\) come from?

From:

$$
\frac{\partial x}{\partial x}=1
$$

That is the mathematical contribution of the **identity shortcut**.

---

# 15. 🧠 Why the \(+1\) Matters

Without the shortcut:

$$
y=F(x)
$$

Then:

$$
\frac{\partial y}{\partial x}
=
\frac{\partial F}{\partial x}
$$

So:

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial F}{\partial x}
$$

The gradient must pass through \(F\).

---

With the shortcut:

$$
y=F(x)+x
$$

we get:

$$
\boxed{
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\left(
\frac{\partial F}{\partial x}+1
\right)
}
$$

So there is an **identity contribution**.

---

# 16. 🔢 Numerical Gradient Example

Suppose:

$$
\frac{\partial L}{\partial y}=1
$$

and:

$$
\frac{\partial F}{\partial x}=-0.9
$$

### Without shortcut

$$
\frac{\partial L}{\partial x}
=
1(-0.9)
$$

$$
=-0.9
$$

### With shortcut

$$
\frac{\partial L}{\partial x}
=
1(-0.9+1)
$$

$$
=\boxed{0.1}
$$

This is only a simplified scalar example, but it makes the important idea visible:

> **The shortcut creates an additional direct gradient pathway.**

---

# 17. ⚠️ Important: Don't Say "ResNet Eliminates Vanishing Gradients"

That's too simplistic.

A technically better statement is:

> **Residual/skip connections provide a direct pathway for information and gradients and make optimization of very deep networks easier.**

The original ResNet paper's central framing was the **degradation problem** and easier optimization of deeper networks. ([arXiv][1])

So remember:

```text
❌ ResNet = magically eliminates all gradient problems

✅ ResNet = provides shortcut pathways that make
   very deep networks easier to optimize
```

---

# 18. 🧩 Dimension Matching

There's an important practical condition:

$$
F(x)+x
$$

requires compatible tensor shapes.

For example:

```text
F(x) = [32, 64, 56, 56]
x    = [32, 64, 56, 56]
```

Addition works:

```text
[32,64,56,56]
       +
[32,64,56,56]
       ↓
[32,64,56,56]
```

---

But:

```text
F(x) = [32,128,28,28]

x = [32,64,56,56]
```

cannot be directly added.

---

# 19. 🔧 Projection Shortcut

When dimensions change, the shortcut can use a transformation such as a **1×1 convolution**.

```text
                    ┌── 1×1 Conv ─────┐
                    │                 ↓
x ──────────────────┤                (+)
                    │                 ↑
                    ↓                 │
                Conv → Conv ──────────┘
```

For example:

```text
Input:
[32,64,56,56]

      ↓ 1×1 Conv

Shortcut:
[32,128,28,28]
```

Now it can be added to:

```text
Residual branch:
[32,128,28,28]
```

The original ResNet architecture uses projection shortcuts when dimensions need to be matched. ([arXiv][1])

---

# 20. 🔢 Shape Example

Suppose:

```text
Input
[32, 64, 56, 56]
```

Residual branch:

```text
Conv
↓
[32, 128, 28, 28]
```

Shortcut:

```text
Input
↓
1×1 Conv, stride=2
↓
[32,128,28,28]
```

Then:

```text
[32,128,28,28]
          +
[32,128,28,28]
          ↓
[32,128,28,28]
```

### 🔥 Rule

For residual addition:

> **The tensors being added must have compatible dimensions.**

---

# 21. 🏗️ Basic ResNet Data Flow

```text
                    INPUT
                      │
              ┌───────┴────────┐
              │                │
              ↓                │
         Conv → BN → ReLU      │
              ↓                │
             Conv              │
              ↓                │
             BN                │
              │                │
              └───────→ (+) ←──┘
                         │
                        ReLU
                         │
                         ↓
                       OUTPUT
```

The exact ordering varies between ResNet variants, so don't memorize this as universal.

The essential structure is:

$$
\boxed{\text{output} = \text{residual branch} + \text{shortcut}}
$$

---

# 22. 🔄 Plain CNN → ResNet

Understand the evolution:

```text
Plain CNN
   ↓
Make network deeper
   ↓
Deeper plain networks become difficult to optimize
   ↓
Degradation problem
   ↓
Residual learning
   ↓
Learn F(x) instead of directly H(x)
   ↓
Shortcut connection
   ↓
F(x) + x
   ↓
Better information and gradient pathways
```

---

# 23. 🆚 Plain Network vs ResNet

| Aspect            | Plain CNN               | ResNet              |
| ----------------- | ----------------------- | ------------------- |
| Main path         | Sequential              | Residual + shortcut |
| Transformation    | Learn \(H(x)\) directly | Learn \(F(x)\)      |
| Shortcut          | ❌                       | ✅                   |
| Identity pathway  | ❌                       | ✅                   |
| Basic equation    | \(y=H(x)\)              | \(y=F(x)+x\)        |
| Deep optimization | More difficult          | Easier              |
| Main innovation   | Deeper stacking         | Residual learning   |

---

# 24. 🧠 The Real Meaning of "Residual"

The residual is:

$$
\boxed{F(x)=H(x)-x}
$$

Think:

```text
Desired transformation
        ↓
       H(x)

Current input
        ↓
        x

Difference
        ↓
    H(x) - x

        ↓

    Residual F(x)
```

So the network learns:

> **How much should I change the current representation?**

---

# 25. 🧠 What If the Desired Transformation Is Almost Identity?

Suppose:

$$
H(x)\approx x
$$

Then:

$$
H(x)-x\approx0
$$

Therefore:

$$
F(x)\approx0
$$

and:

$$
H(x)=F(x)+x\approx x
$$

So the residual branch can learn something close to zero while the shortcut preserves the useful representation.

🔥 This is the key intuition behind residual learning.

---

# 26. 🏭 Why ResNet Was Such an Important Architecture

The original work demonstrated residual networks up to **152 layers** on ImageNet and reported that these networks were easier to optimize than comparable plain networks. ([arXiv][1])

The bigger lesson is architectural:

> **A network doesn't always need every layer to completely transform the representation. Sometimes preserving information and learning corrections is easier.**

This idea became much broader than CNN classification; residual/skip connections became a recurring design pattern in later deep-learning architectures.

---

# 27. 🎯 Architecture Learning Rule

Don't memorize:

```text
Conv → BN → ReLU → Conv → BN → Add
```

as the main idea.

Remember:

```text
PROBLEM
   ↓
Deep plain networks become harder to optimize
   ↓
SOLUTION
   ↓
Residual learning
   ↓
Learn correction F(x)
   ↓
Shortcut carries x
   ↓
F(x) + x
```

This is exactly the architecture-learning approach from your curriculum: **problem → innovation → mechanism → why it helps → evolution**, rather than memorizing layer-by-layer diagrams. 

---

# 🔗 Online Visual Resources

### 🖼️ Original ResNet paper

**[Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)** — the primary source. Use its figures to see the degradation experiment and residual-block design. ([arXiv][1])

### 🔗 Interactive CNN visualization

**[CNN Explainer](https://poloclub.github.io/cnn-explainer/)** — useful for connecting convolution operations, feature maps, and tensor flow.

### 📚 PyTorch implementation reference

**[PyTorch ResNet discussion / BasicBlock](https://discuss.pytorch.org/t/help-in-resnet-basicblock-architecture/102768)** — useful when you later reach implementation and want to understand how shortcut/downsample paths are represented in code. ([PyTorch Forums][2])

---

# 🎤 Interview Questions

### 1. What is the degradation problem?

**Answer:**
As plain CNNs become deeper, their training error can increase rather than decrease, indicating increasing optimization difficulty. ([arXiv][1])

### 2. What is residual learning?

$$
F(x)=H(x)-x
$$

The layers learn the residual correction instead of directly learning the complete transformation.

### 3. What is a skip connection?

A shortcut that bypasses one or more layers and carries the input directly to a later addition operation.

### 4. What is the fundamental ResNet equation?

$$
\boxed{y=F(x)+x}
$$

### 5. Why does the skip connection help?

It provides a direct pathway for information and gradients and makes deep networks easier to optimize.

### 6. Why is there a \(+1\) in the gradient?

Because:

$$
\frac{\partial(x)}{\partial x}=1
$$

for the identity shortcut.

### 7. What happens if input and residual output dimensions differ?

Use a projection shortcut, commonly involving a 1×1 convolution, to make the dimensions compatible.

### 8. Is degradation the same as overfitting?

**No.** The original degradation observation concerns increasing **training error** in deeper plain networks, whereas ordinary overfitting primarily concerns a gap between training and generalization performance.

---

# 🚀 QUICK REVISION CHEAT SHEET

## 🔥 Degradation

```text
More depth
    ↓
Not always better
    ↓
Plain network training error can increase
    ↓
Optimization difficulty
```

## 🧠 Residual Learning

$$
\boxed{F(x)=H(x)-x}
$$

Therefore:

$$
\boxed{H(x)=F(x)+x}
$$

---

## 🔗 Skip Connection

```text
             ┌───────────────┐
             │               │
x ───────────┤               ↓
             │              (+)
             ↓               ↑
          F(x) ──────────────┘
```

---

## 🧩 Residual Block

```text
Input x
  │
  ├───────────────┐
  │               │
  ↓               │
Conv → ReLU → Conv│
  │               │
  └──────→ (+) ←──┘
             ↓
           Output
```

---

## 📐 Gradient

$$
y=F(x)+x
$$

Therefore:

$$
\boxed{
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\left(
\frac{\partial F}{\partial x}+1
\right)
}
$$

The **+1** comes from the identity shortcut.

---

## 🔧 Dimension mismatch

```text
F(x) shape ≠ x shape
        ↓
Projection shortcut
        ↓
Often 1×1 Conv
        ↓
Shapes become compatible
        ↓
Addition
```

---

# 🏆 The 20-second ResNet explanation

If an interviewer asks **"Explain ResNet"**, your mental answer should be:

> **Very deep plain CNNs can suffer from the degradation problem, where increasing depth makes training error worse because optimization becomes difficult. ResNet addresses this through residual learning: instead of directly learning \(H(x)\), a block learns \(F(x)=H(x)-x\), and the output is \(F(x)+x\). The skip connection provides a direct pathway for the original representation and gradients, making very deep networks easier to optimize.**

---